In [4]:
###初始化数据
import pandas as pd
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

import numpy as np
import os
from dotenv import load_dotenv
load_dotenv('密码.env') 

db_user = os.getenv("MYSQL_USER")
db_password = os.getenv("MYSQL_PASSWORD")
db_host = os.getenv("MYSQL_HOST")
db_port = os.getenv("MYSQL_PORT")
db_name = os.getenv("MYSQL_DB")

engine= create_engine(f'mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/tysql')
sql="select * from customers"
#"SELECT 
#    cust_name, 
#    LOWER(TRIM(cust_name)) AS clean_name 
#FROM tysql.customers;

#SELECT 
#    cust_email,
#    SUBSTRING_INDEX(cust_email, '@', -1) AS domain,
#    REPLACE(cust_email, 'john', 'j***') AS masked_email
#FROM tysql.customers;

#SELECT 
#    cust_email,
#    REGEXP_REPLACE(cust_email, '[0-9]', '') AS email_no_digits
#FROM tysql.customers;
#"

TRIM / LOWER 是为了什么？（统一格式，方便后续 JOIN 或去重）。

提取域名在业务上有什么用？（可以分析用户都爱用 QQ 邮箱还是公司邮箱）。

正则脱敏在业务上有什么用？（保护用户隐私，符合安全合规）

In [7]:
df=pd.read_sql(sql,engine)

df['cust_name'] = df['cust_name'].fillna('').astype(str)
df['cust_email'] = df['cust_email'].fillna('').astype(str)
df['clean_name'] = df['cust_name'].str.strip().str.lower()
df['domain'] = df['cust_email'].str.split('@').str[-1]
df['email_no_digits'] = df['cust_email'].str.replace(r'\d+', '', regex=True)
print(df)

      cust_id      cust_name          cust_address cust_city cust_state  \
0  1000000001   Village Toys        200 Maple Lane   Detroit         MI   
1  1000000002     Kids Place  333 South Lake Drive  Columbus         OH   
2  1000000003        Fun4All         1 Sunny Place    Muncie         IN   
3  1000000004        Fun4All   829 Riverside Drive   Phoenix         AZ   
4  1000000005  The Toy Store      4545 53rd Street   Chicago         IL   

  cust_zip cust_country        cust_contact             cust_email  \
0    44444          USA          John Smith  sales@villagetoys.com   
1    43333          USA      Michelle Green                          
2    42222          USA           Jim Jones     jjones@fun4all.com   
3    88888          USA  Denise L. Stephens  dstephens@fun4all.com   
4    54545          USA          Kim Howard                          

      clean_name           domain        email_no_digits  
0   village toys  villagetoys.com  sales@villagetoys.com  
1     kids

## 文本清洗实战总结
1. **去空格与统一大小写**：使用 `.str.strip().str.lower()` 清理用户姓名，避免因大小写或前后空格导致的匹配失败。
2. **按分隔符提取**：使用 `.str.split('@').str[-1]` 提取邮箱域名，可用于后续分析客户偏好的邮箱类型（如企业邮箱 vs 个人邮箱）。
3. **正则替换的隐患**：使用 `r'\d+'` 清洗数字时，发现会误伤域名中的数字（如 `fun4all` 变成 `funall`）。在真实业务中，需使用更精确的正则（如先行断言）或分步清洗，以免破坏原数据。